# Batch correction with Scanorama

Scanorama (Hie, Bryson & Berger, *Nat Biotechnol* 2019) finds mutually-nearest-neighbours between batches and uses them to panorama-stitch the batches into a shared low-dimensional space. Strong when cell-type compositions differ across batches.

This is one of the **omicverse batch-correction zoo** tutorials. For an overview of all backends, the recommendation tree, and the unified `_BATCH_OBSM` schema, see [batch/index](../index.md). For the side-by-side comparison of all backends on the NeurIPS 2021 multi-batch benchmark, see [t_single_batch](../t_single_batch.ipynb).

Optional dependency: `pip install scanorama`.


## Load a multi-batch dataset

We use the same toy multi-batch AnnData as the other zoo tutorials — three NeurIPS 2021 batches concatenated. Replace with your own dataset by changing `adata` and `batch_key` below.

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np

# Replace these URLs with your own dataset; the three are
# the same multi-batch dataset used by t_single_batch.
adata1 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932005',
    filename='neurips2021_s1d3.h5ad',
)
adata2 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932008',
    filename='neurips2021_s2d1.h5ad',
)
adata3 = ov.datasets.get_adata(
    'https://figshare.com/ndownloader/files/41932011',
    filename='neurips2021_s3d7.h5ad',
)
adata = sc.concat([adata1, adata2, adata3], merge='same')
adata.obs['batch'] = adata.obs['batch'].astype('category')
adata

## Preprocess + PCA (shared across all backends)

Every backend in the zoo starts from the same QC'd, log-normalised AnnData with `scaled|original|X_pca` in obsm. Read [t_single_batch](../t_single_batch.ipynb) for the full discussion of these steps.

In [ ]:
adata = ov.pp.qc(adata,
                 tresh={'mito_perc': 0.2, 'nUMIs': 500,
                        'detected_genes': 250})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson',
                         n_HVGs=2000, batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=50)

## Run `ov.single.batch_correction(methods='scanorama')`

The wrapper routes method-specific kwargs to the right destination — for scvi-tools backends this includes splitting between `__init__` (architecture) and `.train()` (optimisation). See the **Key parameters** section below.

In [ ]:
ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='scanorama',
)

## Visualise the corrected embedding

Every backend writes its corrected representation to a stable obsm key — for **Scanorama** it is `adata.obsm['X_scanorama']`. We project it via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_scanorama'] = ov.utils.mde(adata.obsm['X_scanorama'])
ov.pl.embedding(
    adata,
    basis='X_mde_scanorama',
    color=['batch'],
    frameon='small',
    title='Scanorama — coloured by batch',
)

## Key parameters

- `n_pcs` — embedding dimension (default 50).
- Scanorama internally requires raw counts in `.X` for some operations; check `adata.X.min()` before calling.


## Related tutorials

- `harmony` — faster, but assumes compositions are roughly comparable across batches.
- `scVI` — deep-learning alternative for the same scenario.

For a side-by-side comparison of every backend on the same benchmark + scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).